In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-rag-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

# RAG A/B 비교 실험

**목적**: chunk_size / top_k 설정에 따라 정확도가 어떻게 달라지는지 비교

| 실험 | chunk_size | chunk_overlap | top_k |
|------|-----------|--------------|-------|
| v1 (small) | 300 | 50 | 2 |
| v2 (medium) | 800 | 120 | 4 |
| v3 (large) | 1500 | 200 | 6 |

# 1. 문서 로드 — 전체 문서 사용

In [ ]:
PDF_PATHS = [
    PROJECT_ROOT / "data/raw/pdf/saving_tips/1.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/2.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/3.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/4.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/5.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/6.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/7.pdf",
    PROJECT_ROOT / "data/raw/pdf/kca_report/0417_kca_split.pdf",
    PROJECT_ROOT / "data/raw/pdf/welfare/2026_hope_ladder_selected.pdf",
    PROJECT_ROOT / "data/raw/pdf/self_report/consumer_spending_report_v2.pdf",
    PROJECT_ROOT / "data/raw/txt/saving_guides.txt",
]

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader, TextLoader

all_docs = []
for path in PDF_PATHS:
    try:
        if str(path).endswith(".pdf"):
            loader = PDFPlumberLoader(str(path))
        elif str(path).endswith(".txt"):
            loader = TextLoader(str(path), encoding="utf-8")
        else:
            continue
        docs = loader.load()
        for d in docs:
            d.metadata["source"] = path.name
        all_docs.extend(docs)
    except Exception as e:
        print(f"로드 실패: {path.name} — {e}")

print("총 문서 수:", len(all_docs))

# 2. 설정별 vectorstore 빌더

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def build_retriever(chunk_size, chunk_overlap, top_k):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    split_docs = splitter.split_documents(all_docs)
    vs = FAISS.from_documents(split_docs, embeddings)
    print(f"  chunk={chunk_size}, overlap={chunk_overlap}, top_k={top_k} → 벡터 {vs.index.ntotal}개")
    return vs.as_retriever(search_kwargs={"k": top_k})

def make_target(retriever):
    def target(inputs: dict):
        q = inputs["question"]
        docs = retriever.invoke(q)
        context_texts = [doc.page_content[:400] for doc in docs]
        answer = llm.invoke(
            f"""질문: {q}

아래 문서에 있는 내용만 사용해서 핵심 답변을 1~2문장으로 작성하세요.
문서에 없는 내용은 절대 추가하지 마세요.

문서:
{context_texts}"""
        ).content
        return {"answer": answer, "contexts": context_texts}
    return target

print("빌더 함수 준비 완료")

In [ ]:
# 3가지 설정으로 retriever 생성
print("retriever 생성 중...")
retriever_v1 = build_retriever(chunk_size=300,  chunk_overlap=50,  top_k=2)
retriever_v2 = build_retriever(chunk_size=800,  chunk_overlap=120, top_k=4)
retriever_v3 = build_retriever(chunk_size=1500, chunk_overlap=200, top_k=6)

target_v1 = make_target(retriever_v1)
target_v2 = make_target(retriever_v2)
target_v3 = make_target(retriever_v3)

print("완료")

# 3. LangSmith Dataset 확인 / 생성

이미 만든 `catcher-rag-welfare-eval` 데이터셋을 재사용합니다.

In [ ]:
from langsmith import Client

client = Client()

# 기존 welfare 데이터셋 재사용 (이미 있으면)
dataset_name = "catcher-rag-welfare-eval"

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    print(f"기존 dataset 재사용: {dataset_name}")
else:
    # 없으면 새로 생성
    questions = [
        "여성청소년 생리용품 지원의 월 지원금은 얼마인가?",
        "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 얼마인가?",
        "저소득 청소년부모 아동양육비 지원의 월 지원금은 얼마인가?",
        "3~5세 유치원 학비 중 국·공립유치원 교육비는 월 얼마인가?",
        "청년내일저축계좌의 정부 매칭 한도는 얼마인가?",
    ]
    ground_truths = [
        "여성청소년 생리용품 지원의 월 지원금은 1만 4,000원이다.",
        "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 13만 원이다.",
        "저소득 청소년부모 아동양육비 지원의 월 지원금은 25만 원이다.",
        "3~5세 유치원 학비 중 국·공립유치원 교육비는 월 10만 원이다.",
        "청년내일저축계좌의 정부 매칭 한도는 월 최대 30만 원이며, 3년 만기 시 최대 1,440만 원 적립 가능하다.",
    ]
    dataset = client.create_dataset(dataset_name=dataset_name, description="복지정책 RAG A/B 비교용")
    for q, gt in zip(questions, ground_truths):
        client.create_example(inputs={"question": q}, outputs={"ground_truth": gt}, dataset_id=dataset.id)
    print(f"새 dataset 생성: {dataset_name}")

# 4. Evaluator 정의

In [ ]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def correctness_evaluator(run, example):
    answer = run.outputs["answer"]
    ground_truth = example.outputs["ground_truth"]
    prompt = f"""다음 답변이 정답과 얼마나 일치하는지 0~1 점수로 평가해줘.
핵심 수치(금액, 기간 등)가 맞으면 높게, 틀리면 낮게.
정답: {ground_truth}
답변: {answer}
숫자 하나만 출력해."""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "correctness", "score": float(score)}

def faithfulness_evaluator(run, example):
    answer = run.outputs["answer"]
    contexts = run.outputs["contexts"]
    prompt = f"""아래 답변이 문서에 있는 내용만 사용했는지 0~1로 평가해줘.
숫자 하나만 출력해.
문서: {contexts}
답변: {answer}"""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "faithfulness", "score": float(score)}

import re
def contains_amount_evaluator(run, example):
    answer = run.outputs["answer"]
    has_amount = bool(re.search(r'\d[\d,]*\s*(원|만원|만\s*원)', answer))
    return {"key": "contains_amount", "score": 1 if has_amount else 0}

print("evaluator 3개 정의 완료")

# 5. A/B/C 실험 순차 실행 → LangSmith에서 Compare

실행 후 LangSmith → Datasets → `catcher-rag-welfare-eval` → **Compare** 탭에서 3개 실험을 나란히 비교할 수 있습니다.

In [ ]:
from langsmith.evaluation import evaluate

evaluators = [correctness_evaluator, faithfulness_evaluator, contains_amount_evaluator]

print("=== v1: chunk=300, top_k=2 ===")
evaluate(target_v1, data=dataset_name, evaluators=evaluators, experiment_prefix="ab-chunk300-k2")

In [ ]:
print("=== v2: chunk=800, top_k=4 ===")
evaluate(target_v2, data=dataset_name, evaluators=evaluators, experiment_prefix="ab-chunk800-k4")

In [ ]:
print("=== v3: chunk=1500, top_k=6 ===")
evaluate(target_v3, data=dataset_name, evaluators=evaluators, experiment_prefix="ab-chunk1500-k6")

# 6. 결과 요약

LangSmith → Datasets → `catcher-rag-welfare-eval` → Experiments 탭에서
`ab-chunk300-k2`, `ab-chunk800-k4`, `ab-chunk1500-k6` 세 실험을 선택 후 **Compare** 클릭

**판단 기준:**
- correctness 높은 설정 → 정답 일치율 좋음
- faithfulness 높은 설정 → 문서 범위 내 답변 잘 함
- 두 지표가 모두 높은 설정을 프로덕션에 적용